# SpOC-3 — Spacekangaroos on GPU

This notebook runs **one thing**: `cuda-torso`, the actual winning entry, on a
GPU. Everything else (hill climbing, Team HRI, fast-cma-es, min-degree) runs on
your Mac via `run_mac.sh`, four solvers in parallel, one core each.

Results here are re-scored through the same `esa_eval.py` the Mac solvers use,
so the columns are comparable.

## Run it

1. *Add Input* → Datasets → your **hc-simple** dataset
2. *Accelerator* → **GPU T4 x2**
3. *Save Version* → **Save & Run All (Commit)**
4. When it finishes: Output → download `benchmark-spacekangaroos.csv`

Nothing below needs editing — `SOURCE_DIR` is already set for a dataset named
`hc-simple`. Takes about 10 hours and uses 10 of your 30 weekly GPU hours.


In [ ]:
# ─────────────────────────── CONFIG ───────────────────────────
# Nothing here needs changing if your dataset is named "hc-simple".

SOLVERS = ["spacekangaroos"]     # GPU only. CPU solvers run on the Mac.

DATA_DIR        = "data/v2"      # twin-preserving instances
SECONDS_PER_RUN = 1200           # 20 min per (instance, seed) -- must match
SEEDS           = 3              # what run_mac.sh uses on the Mac
INSTANCES       = ["small-graph", "medium-graph", "large-graph"] + \
                  [f"synth-{i}" for i in range(1, 8)]

REPO_URL    = ""                 # unused; we read from the attached dataset
REPO_BRANCH = "hill-climbing-simple"
RESUME_GLOB = "/kaggle/input/**/benchmark*.csv"

WORK    = "/kaggle/working/hc-simple"
OUT_CSV = "/kaggle/working/benchmark-" + "-".join(SOLVERS) + ".csv"

# Find the dataset wherever Kaggle mounted it, whatever you named it: look
# for the folder that actually contains torso.py. Hardcoding a path just
# means a 10-second failure when the dataset slug differs by a character.
import glob as _glob, os
cands = [os.path.dirname(p)
         for p in _glob.glob("/kaggle/input/**/torso.py", recursive=True)]
if not cands:
    listing = sorted(_glob.glob("/kaggle/input/*")) or ["(nothing attached)"]
    raise SystemExit(
        "Could not find torso.py under /kaggle/input.\n"
        "Attach the hc-simple dataset: right panel -> Add Input -> Datasets.\n"
        "What is attached right now:\n  " + "\n  ".join(listing))
SOURCE_DIR = cands[0]
print(f"dataset found at: {SOURCE_DIR}")

est = len(INSTANCES) * len(SOLVERS) * SEEDS * SECONDS_PER_RUN / 3600
print(f"{len(INSTANCES)} instances x {SEEDS} seeds x {SECONDS_PER_RUN}s "
      f"= {est:.1f} h")
print(f"writing -> {OUT_CSV}")
if est > 10.5:
    print("WARNING: close to the 12-hour session cap.")


In [ ]:
# ─────────────────────── SETUP: code, data, CUDA ───────────────────────
import os, shutil, subprocess, sys, glob

if SOURCE_DIR:
    if os.path.exists(WORK): shutil.rmtree(WORK)
    shutil.copytree(SOURCE_DIR, WORK)
    print(f"copied {SOURCE_DIR} -> {WORK}")
else:
    if not os.path.exists(WORK):
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH,
                        REPO_URL, WORK], check=True)
    print(f"cloned {REPO_BRANCH}")

os.chdir(WORK)
sys.path.insert(0, WORK)
sys.path.insert(0, os.path.join(WORK, "leaderboard-references"))

missing = [i for i in INSTANCES if not os.path.exists(f"{DATA_DIR}/{i}.gr")]
assert not missing, f"missing instance files: {missing}"
print(f"{len(INSTANCES)} instances present")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fcmaes"],
               check=False)

# GPU only: compile the winning entry's CUDA evaluator and teach it our
# instances. GRAPH_SIZES is what limits it to the three official graphs.
GPU_READY = False
if "spacekangaroos" in SOLVERS:
    cuda = os.path.join(WORK, "leaderboard-references", "cuda-torso")
    os.makedirs(f"{cuda}/data", exist_ok=True)
    sizes = {}
    for name in INSTANCES:
        mx = 0
        with open(f"{WORK}/{DATA_DIR}/{name}.gr") as f:
            for line in f:
                p = line.split()
                if len(p) == 2 and p[0] != "p":
                    mx = max(mx, int(p[0]), int(p[1]))
        sizes[name] = mx + 1
        for sub in ("logs", "submissions", "checkpoints"):
            os.makedirs(f"{cuda}/{sub}/{name}", exist_ok=True)
        shutil.copy(f"{WORK}/{DATA_DIR}/{name}.gr", f"{cuda}/data/{name}.gr")

    src = open(f"{cuda}/run.py").read()

    # run.py torch.save()s its whole elite tensor every 50 generations,
    # into a checkpoints/ folder. We only ever read its submission JSONs,
    # and those .pt files pile up into gigabytes over a 20-minute run.
    # Skip the state dump -- it is I/O for resuming, not part of the
    # algorithm. (Leaving it in is also what crashed the first attempt.)
    _mark = "checkpoint_path = f" + chr(34) + "checkpoints/"
    if _mark in src:
        head, tail = src.split(_mark, 1)
        tail = tail.split("def create_submission", 1)[1]
        src = head + "pass\n\n\ndef create_submission" + tail
        print("disabled run.py state checkpointing")

    start = src.index("GRAPH_SIZES = {")
    end   = src.index("}", start) + 1
    src = src[:start] + "GRAPH_SIZES = " + repr(sizes) + src[end:]
    open(f"{cuda}/run.py", "w").write(src)
    GRAPH_N = dict(sizes)
    print("patched GRAPH_SIZES:", sizes)

    # Fix a 32-bit overflow in their kernel. adj_offset indexes a B x N x N
    # bool array but is declared int, so idx * N * N wraps negative once it
    # passes 2^31: at n=2426 that is thread 365 of 1024, and every n=2426 run
    # died with "illegal memory access". n=1357 and n=1399 stay under the
    # limit at B=1024, which is exactly the split we observed. Widening the
    # type changes no arithmetic -- it just stops the wrap.
    cu = open(f"{cuda}/libeval.cu").read()
    if "int adj_offset" in cu:
        cu = cu.replace("int adj_offset = idx * N * N;",
                        "size_t adj_offset = idx * N * N;")
        open(f"{cuda}/libeval.cu", "w").write(cu)
        print("patched libeval.cu: adj_offset int -> size_t")

    r = subprocess.run("nvcc -Xcompiler -fPIC -shared -o libeval.so libeval.cu",
                       shell=True, cwd=cuda, capture_output=True, text=True)
    if r.returncode == 0:
        GPU_READY = True
        print("compiled libeval.so")
    else:
        print("CUDA COMPILE FAILED — is the accelerator set to GPU?")
        print(r.stderr[-2000:])

In [ ]:
# ─────────────── COMMON: one evaluator, one CSV, resume ───────────────
import csv, json, random, time
from esa_eval import build_adj_bitsets, evaluate as esa_evaluate, \
                     hypervolume_2d, MAX_TW
from torso import Graph

FIELDS = ["instance", "n", "solver", "seed", "score", "seconds", "valid"]

def score_vectors(graph, vectors):
    """Re-score ANY solver's decision vectors through esa_eval. (ok, score)."""
    n = graph.n
    bits = build_adj_bitsets(n, graph.adj)
    pts = []
    for vec in vectors:
        perm, t = [int(x) for x in vec[:-1]], int(vec[-1])
        if len(vec) != n + 1 or sorted(perm) != list(range(n)):  return False, 0
        if not 0 <= t < n:                                        return False, 0
        w, t = esa_evaluate(perm, t, bits, n)
        if w > MAX_TW:                                            return False, 0
        pts.append((w, t))
    if len(pts) > 20 or len(pts) != len(set(pts)):                return False, 0
    return True, -int(hypervolume_2d(pts, n))

prior = sorted(glob.glob(RESUME_GLOB, recursive=True)) if RESUME_GLOB else []
if os.path.exists(OUT_CSV): prior.append(OUT_CSV)

done = set()
rows_kept = []
for path in prior:
    with open(path) as f:
        for r in csv.DictReader(f):
            key = (r["instance"], r["solver"], int(r["seed"]))
            if key in done:
                continue
            # A row is worth keeping if it is valid, OR if it is invalid but
            # actually spent the budget -- that is a real "found nothing"
            # result. An invalid row that finished in seconds is a crash, and
            # crashes get retried rather than frozen into the table.
            ok = str(r.get("valid", "True")).lower() == "true"
            spent = float(r.get("seconds", 0)) >= 0.9 * SECONDS_PER_RUN
            if ok or spent:
                done.add(key); rows_kept.append(r)
            else:
                print(f"  retrying {r['instance']} seed {r['seed']}: "
                      f"invalid after only {float(r['seconds']):.0f}s")

csv_file = open(OUT_CSV, "w", newline="")
writer = csv.DictWriter(csv_file, fieldnames=FIELDS)
writer.writeheader()
for r in rows_kept:
    writer.writerow({k: r.get(k, "") for k in FIELDS})
csv_file.flush()
print(f"carried forward {len(done)} runs from {len(prior)} file(s)")
mine = sum(1 for (_, sv, _) in done if sv in SOLVERS)
print(f"  of those, {mine} are for this session's solver(s) and will be skipped")

def record(inst, n, solver, seed, score, secs, valid):
    writer.writerow({"instance": inst, "n": n, "solver": solver, "seed": seed,
                     "score": score, "seconds": round(secs, 1), "valid": valid})
    csv_file.flush()
    done.add((inst, solver, seed))
    print(f"   {solver:<15} seed {seed}: {score:>14,}  "
          f"({secs/60:.1f} min){'' if valid else '  INVALID'}", flush=True)

## Spacekangaroos — the real winning entry

`cuda-torso` evolves 1024 candidates per generation on the GPU and checkpoints
a submission every 50 generations. We stop it at the budget, take its best
checkpoint and **re-score it with `esa_eval.py`**, so it is measured exactly
like the Mac solvers rather than trusted.

It selects its 20 points with a *greedy* rule while our solvers use the exact
HSSP, so this column is if anything a slight **under**-estimate.

In [ ]:
# ───────────────── GPU: Spacekangaroos = real cuda-torso ─────────────────
import signal

def run_cuda_torso(name, seconds, seed):
    """Run the actual entry, then re-score its best checkpoint."""
    cuda = os.path.join(WORK, "leaderboard-references", "cuda-torso")
    for old in glob.glob(f"{cuda}/submissions/{name}/*.json"):
        os.remove(old)
    env = dict(os.environ, PYTHONHASHSEED=str(seed))

    # run.py holds adjs AND bkup_adjs, each a B x N x N bool tensor on the
    # GPU. At B=1024 that is 12.1 GB for n=2426 -- more than a T4 has, which
    # is why large-graph, synth-6 and synth-7 died in 10 seconds while every
    # n=1357/1399 instance ran the full budget. Size the batch to fit.
    n = GRAPH_N[name]
    fit = int(8e9 / (2 * n * n))
    batch = max(64, min(1024, 1 << (fit.bit_length() - 1)))
    if batch < 1024:
        print(f"   batch {batch} (not 1024): {2*batch*n*n/1e9:.1f} GB of GPU "
              f"memory at n={n}", flush=True)

    t0 = time.time()
    p = subprocess.Popen([sys.executable, "run.py", "--graph", name,
                          "--batch_size", str(batch)],
                         cwd=cuda, env=env,
                         stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    crashed = False
    try:
        p.wait(timeout=seconds)
        # Finishing EARLY means it died -- max_generations is 100,000 and it
        # will never reach that in the budget. Say so loudly; a silent crash
        # here is how you end up benchmarking generation zero.
        if time.time() - t0 < seconds * 0.9:
            crashed = True
            err = (p.stderr.read() or b"").decode()[-1500:]
            print(f"   !! run.py exited after {time.time()-t0:.0f}s of "
                  f"{seconds:.0f}s -- IT CRASHED", flush=True)
            print("   " + err.replace(chr(10), chr(10) + "   "), flush=True)
    except subprocess.TimeoutExpired:
        p.send_signal(signal.SIGINT)
        try: p.wait(timeout=60)
        except subprocess.TimeoutExpired: p.kill()

    best = (False, 0)
    graph = Graph.load(f"{WORK}/{DATA_DIR}/{name}.gr")
    for path in glob.glob(f"{cuda}/submissions/{name}/*.json"):
        try:
            vecs = json.load(open(path))["decisionVector"]
        except Exception:
            continue
        ok, sc = score_vectors(graph, vecs)
        if ok and sc < best[1]:
            best = (True, sc)
    return best

if "spacekangaroos" in SOLVERS:
    if not GPU_READY:
        print("SKIPPING: libeval.so did not compile. Set accelerator to GPU.")
    else:
        for name in INSTANCES:
            graph = Graph.load(f"{WORK}/{DATA_DIR}/{name}.gr")
            print(f"{name} (n={graph.n})")
            for seed in range(1, SEEDS + 1):
                if (name, "spacekangaroos", seed) in done:
                    print(f"   spacekangaroos  seed {seed}: already done"); continue
                t0 = time.time()
                ok, sc = run_cuda_torso(name, SECONDS_PER_RUN, seed)
                record(name, graph.n, "spacekangaroos", seed, sc,
                       time.time() - t0, ok)

In [ ]:
# ─────────────── CPU: fast-cma-es, Team HRI, Hill Climbing ───────────────
import hill_climbing
try:    import hri_lns
except Exception as e: hri_lns = None; print("hri unavailable:", e)
try:    import cmaes
except Exception as e: cmaes = None; print("cmaes unavailable:", e)

def run_cpu(solver, graph, seconds, seed):
    if solver == "min_degree":
        front = hill_climbing.min_degree_only(graph, seed)
        return score_vectors(graph, front.decision_vectors())
    if solver == "hill_climbing":
        front, _ = hill_climbing.solve(graph, seconds, seed=seed,
                                       start="min_degree")
    elif solver == "hri":
        front, _, _ = hri_lns.solve(graph, seconds, seed=seed,
                                    start="min_degree")
    elif solver == "fast_cma_es":
        front, _, _ = cmaes.solve(graph, seconds, seed=seed)
    else:
        raise ValueError(solver)
    return score_vectors(graph, front.decision_vectors())

cpu_solvers = [s for s in SOLVERS if s != "spacekangaroos"]
for name in INSTANCES:
    if not cpu_solvers: break
    graph = Graph.load(f"{WORK}/{DATA_DIR}/{name}.gr")
    print(f"{name} (n={graph.n}, edges={graph.edge_count})")
    for solver in cpu_solvers:
        if solver == "hri" and hri_lns is None: continue
        if solver == "fast_cma_es" and cmaes is None: continue
        for seed in range(1, SEEDS + 1):
            if (name, solver, seed) in done:
                print(f"   {solver:<15} seed {seed}: already done"); continue
            t0 = time.time()
            ok, sc = run_cpu(solver, graph, SECONDS_PER_RUN, seed)
            record(name, graph.n, solver, seed, sc, time.time() - t0, ok)

csv_file.flush()
print("\nsession complete — benchmark.csv is in /kaggle/working/")

## Result

Download `benchmark-spacekangaroos.csv` from the Output tab, drop it into
`~/Desktop/SpOC3/_hc_readme/kaggle/` next to the Mac CSVs, then run
`merge_results.py` to get the finished table.

In [ ]:
# ───────────────────────────── SUMMARY ─────────────────────────────
import statistics
from collections import defaultdict

COLS  = ["fast_cma_es", "hri", "spacekangaroos", "hill_climbing"]
LABEL = {"fast_cma_es": "fast-cma-es", "hri": "Team HRI",
         "spacekangaroos": "Spacekangaroos", "hill_climbing": "Hill Climbing"}

scores = defaultdict(list)
seen = set()
for path in (sorted(glob.glob(RESUME_GLOB, recursive=True)) if RESUME_GLOB else []) \
            + [OUT_CSV]:
    if not os.path.exists(path): continue
    with open(path) as f:
        for r in csv.DictReader(f):
            key = (r["instance"], r["solver"], r["seed"])
            if key in seen: continue
            seen.add(key)
            scores[(r["instance"], r["solver"])].append(int(r["score"]))

spreads = [max(v) - min(v) for v in scores.values() if len(v) > 1]
noise = statistics.median(spreads) if spreads else 0

print("instance,"+",".join(LABEL[c] for c in COLS)+",Gap from Best")
for name in INSTANCES:
    vals = {c: min(scores[(name, c)]) for c in COLS if (name, c) in scores}
    if not vals: continue
    live = [v for v in vals.values() if v != 0]
    best = min(live) if live else 0
    hc   = vals.get("hill_climbing")
    gap  = (hc - best) if hc is not None else ""
    print(name + "," + ",".join(str(vals.get(c, "")) for c in COLS) + f",{gap}")

print(f"\nseed spread (median): {noise:,.0f} HV — gaps smaller than this are ties")
missing = [(i, c) for i in INSTANCES for c in COLS if (i, c) not in scores]
if missing:
    print(f"still to run: {len(missing)} cells -> {sorted(set(c for _, c in missing))}")